[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/Open-Athena/MarinFold/blob/main/notebooks/short_document_bias.ipynb)

# Does contacts-v1 generate "too-short" documents? — MarinFold [#142](https://github.com/Open-Athena/MarinFold/issues/142)

contacts-v1 predicts residue–residue contacts by **generating a document** of `<contact> <pI> <pJ>`
statements from the sequence alone. A natural worry: does the model **stop early** and emit **far
fewer contacts than the structure has**, capping recall no matter how good the inference wrapper is?

This notebook **generates rollouts** from the model for a handful of eval proteins and **analyzes**
them: how many contacts each document asserts vs the ground-truth count (`pred/gt`), whether the
documents finish on their own (`<end>`) or get truncated, and how that relates to accuracy.

**What to look for.** Across proteins you should see `pred/gt` land a bit under 1 (mild-to-moderate
under-generation), **100% finished** (the model chooses `<end>` — a generous token budget never
truncates), and the shortfall **biggest exactly where recall is worst** — i.e. short documents are a
*symptom* of the model being unsure of the fold, strong only on hard/long proteins, rather than a
decoding bug.

Runs on a GPU — a Colab **T4** is enough. Model: the exp75 E8 1.5B (`step-35679`, eval loss 2.7566);
switchable to the newer default `contacts-v1-exp120-1.5B`.

In [ ]:
# @title Configuration
MODEL_NICKNAME = "contacts-v1-exp75-1.5B"  # @param ["contacts-v1-exp75-1.5B", "contacts-v1-exp120-1.5B"]
# A short / confident-long / lost-long trio spans the length range and shows the symptom contrast.
# Add any of the 12 eval entry_ids (see the targets table printed below).
PROTEINS = "denovo_pdb__1uw1_A, denovo_pdb__6reo_A, foldbench100__8bfy_A"  # @param {type:"string"}
N_ROLLOUTS = 60  # @param {type:"integer"}
TEMPERATURE = 1.0  # @param {type:"number"}
TOP_P = 0.95  # @param {type:"number"}
TOP_K = 50  # @param {type:"integer"}
print(dict(model=MODEL_NICKNAME, proteins=[x.strip() for x in PROTEINS.split(",")],
           n_rollouts=N_ROLLOUTS, temperature=TEMPERATURE, top_p=TOP_P, top_k=TOP_K))

In [ ]:
# Setup: clone MarinFold + install marinfold[transformers] (build_document + model registry),
# and fetch the eval proteins (sequence + resolved ground-truth contacts). marinfold pins
# transformers<5 so the Levanter rope config loads correctly.
import importlib, importlib.util, io, subprocess, sys, urllib.request
from pathlib import Path
import numpy as np, pandas as pd, pyarrow.parquet as pq

REPO = Path("/content/MarinFold") if Path("/content").exists() else Path.cwd() / "MarinFold"
if not (REPO / "marinfold" / "pyproject.toml").exists():
    subprocess.run(["git", "clone", "--depth", "1",
                    "https://github.com/Open-Athena/MarinFold.git", str(REPO)], check=True)
if importlib.util.find_spec("marinfold") is None:
    subprocess.run([sys.executable, "-m", "pip", "install", "-q",
                    "-e", f"{REPO / 'marinfold'}[transformers]"], check=True)
    if str(REPO / "marinfold") not in sys.path:
        sys.path.insert(0, str(REPO / "marinfold"))
    importlib.invalidate_caches()

RAW = "https://raw.githubusercontent.com/Open-Athena/MarinFold/06014e29c1f08cd7adea29f9d821ec0ac829c6c8/experiments/exp142_evals_short_document_bias"
def fetch_parquet(name):
    with urllib.request.urlopen(f"{RAW}/{name}") as r:
        return pq.read_table(io.BytesIO(r.read())).to_pandas()

targets = fetch_parquet("targets.parquet").set_index("entry_id")
print("marinfold ready; eval proteins:")
print(targets.reset_index()[["entry_id", "dataset", "L", "n_gt"]].sort_values("L").to_string(index=False))

## Generate rollouts

For each protein we build `N_ROLLOUTS` fresh contacts-v1 document realizations (resampled N-terminus
start + statement order — the format's nuisance symmetries), sample a completion of the contact
section, and record per rollout: contacts emitted (`n_pred`, deduped, sep≥6), whether it finished
(`<end>`), the document length in tokens, and recall/precision against the ground truth. The token
budget is deliberately generous (`min(8192−prefix, 6·L+128)`) so a short document can only be the
model's own choice.

In [ ]:
import re, time
import torch
from transformers import AutoModelForCausalLM, AutoTokenizer
from marinfold.registry import resolve_model
from marinfold.document_structures.contacts_v1 import (
    GenerationConfig, build_document, residues_from_sequence,
)

BEGIN, NUM_POS, MIN_SEP = "<begin_statements>", 2000, 6
CONTACT_RE = re.compile(r"<contact>\s+<p(\d+)>\s+<p(\d+)>")

device = "cuda" if torch.cuda.is_available() else "cpu"
dtype = (torch.bfloat16 if device == "cuda" and torch.cuda.is_bf16_supported()
         else torch.float16 if device == "cuda" else torch.float32)

model_dir = resolve_model(MODEL_NICKNAME)          # downloads from the public bucket (anonymous)
tok = AutoTokenizer.from_pretrained(str(model_dir))
if tok.pad_token_id is None:
    tok.pad_token = "<end>"
model = AutoModelForCausalLM.from_pretrained(str(model_dir), dtype=dtype).to(device).eval()
END = tok.convert_tokens_to_ids("<end>")
print(f"loaded {MODEL_NICKNAME} on {device} ({dtype}); vocab={model.config.vocab_size}")


def prefix_and_map(entry, seq, r):
    """One fresh contacts-v1 realization (resampled N-term + statement order)."""
    doc = build_document(f"{entry}:r{r}", residues_from_sequence(seq), [], config=GenerationConfig())
    prefix = doc.document[: doc.document.index(BEGIN) + len(BEGIN)]
    pos2seq = {(doc.n_term_index + i) % NUM_POS: i for i in range(doc.seq_len)}
    return prefix, pos2seq


@torch.no_grad()
def rollouts_for(entry, n, batch=16 if torch.cuda.is_available() else 4):
    row = targets.loc[entry]
    seq, L, n_gt = row["sequence"], int(row["L"]), int(row["n_gt"])
    gt = {(int(i), int(j)) for i, j in row["gt_contacts"]}
    built = [prefix_and_map(entry, seq, r) for r in range(n)]
    plen = len(tok(built[0][0], add_special_tokens=False).input_ids)
    max_new = min(8192 - plen, 6 * L + 128)        # generous: the cap never binds
    recs = []
    for s in range(0, n, batch):
        chunk = built[s:s + batch]
        enc = tok([c[0] for c in chunk], add_special_tokens=False,
                  return_tensors="pt", padding=True).to(device)
        out = model.generate(input_ids=enc["input_ids"], attention_mask=enc["attention_mask"],
                             do_sample=True, temperature=TEMPERATURE, top_p=TOP_P, top_k=TOP_K,
                             max_new_tokens=max_new, eos_token_id=END, pad_token_id=END)
        for (prefix, pos2seq), g in zip(chunk, out[:, enc["input_ids"].shape[1]:].tolist()):
            finished = END in g
            g = g[: g.index(END) + 1] if finished else g
            pred = set()
            for a, b in CONTACT_RE.findall(tok.decode(g, skip_special_tokens=False)):
                ia, ib = pos2seq.get(int(a)), pos2seq.get(int(b))
                if ia is None or ib is None or abs(ia - ib) < MIN_SEP:
                    continue
                pred.add((min(ia, ib), max(ia, ib)))
            tp = len(pred & gt)
            recs.append(dict(entry=entry, L=L, n_gt=n_gt, n_pred=len(pred), finished=finished,
                             n_gen_tokens=len(g), recall=tp / len(gt) if gt else np.nan,
                             precision=tp / len(pred) if pred else 0.0))
    return recs


chosen = [p.strip() for p in PROTEINS.split(",") if p.strip()]
rows = []
for e in chosen:
    t0 = time.time()
    rs = rollouts_for(e, N_ROLLOUTS)
    rows += rs
    pg = np.mean([x["n_pred"] for x in rs]) / rs[0]["n_gt"]
    print(f"  {e:<24} L={rs[0]['L']:>3} n_gt={rs[0]['n_gt']:>3}  pred/gt={pg:.2f}  "
          f"finished={np.mean([x['finished'] for x in rs]):.2f}  "
          f"recall={np.nanmean([x['recall'] for x in rs]):.2f}  ({time.time()-t0:.0f}s)")
roll = pd.DataFrame(rows)
roll["pred_gt"] = roll.n_pred / roll.n_gt

## Analysis

In [ ]:
p, r = roll.precision, roll.recall
roll["f1"] = np.where((p + r) > 0, 2 * p * r / (p + r), 0.0)

agg = (roll.groupby(["entry", "L", "n_gt"])
           .agg(rollouts=("n_pred", "size"), emitted=("n_pred", "mean"), pred_gt=("pred_gt", "mean"),
                frac_lt_half=("pred_gt", lambda s: (s < 0.5).mean()), tokens=("n_gen_tokens", "mean"),
                finished=("finished", "mean"), recall=("recall", "mean"),
                precision=("precision", "mean"), best_f1=("f1", "max"))
           .reset_index().sort_values("L"))
pd.set_option("display.width", 140)
print(agg.round(3).to_string(index=False))

print(f"\npooled over {len(roll)} rollouts:  pred/gt mean {roll.pred_gt.mean():.2f} "
      f"(median {roll.pred_gt.median():.2f})   finished {100*roll.finished.mean():.0f}%   "
      f"rollouts emitting < half the GT count {100*(roll.pred_gt < 0.5).mean():.0f}%")
if len(agg) >= 3:
    print(f"corr(pred/gt, recall) across proteins = {np.corrcoef(agg.pred_gt, agg.recall)[0,1]:+.2f} "
          "  (positive => under-generation tracks accuracy, i.e. a symptom not a decoding bug)")

In [ ]:
import matplotlib.pyplot as plt

order = agg.entry.tolist()
Ls = agg.L.values
fig, ax = plt.subplots(2, 2, figsize=(13, 9))
wid = max(6, (Ls.max() - Ls.min()) / 25) if len(Ls) > 1 else 8

# (1) pred/gt distribution per protein vs L
groups = [roll.loc[roll.entry == e, "pred_gt"].values for e in order]
bp = ax[0,0].boxplot(groups, positions=Ls, widths=wid, showfliers=False,
                     patch_artist=True, manage_ticks=False)
for b in bp["boxes"]: b.set(facecolor="#4C72B0", alpha=0.5)
ax[0,0].axhline(1.0, color="crimson", ls="--", lw=1.5, label="parity (pred = GT count)")
ax[0,0].axhline(0.5, color="gray", ls=":", label="half of GT")
ax[0,0].set(xlabel="protein length L", ylabel="contacts emitted / n_gt", title="Contacts emitted vs ground truth")
ax[0,0].set_ylim(0, max(1.6, min(3.0, roll.pred_gt.max()))); ax[0,0].legend(fontsize=8)

# (2) mean emitted vs n_gt (parity)
sc = ax[0,1].scatter(agg.n_gt, agg.emitted, c=Ls, cmap="viridis", s=80, zorder=3)
mx = max(agg.n_gt.max(), agg.emitted.max()) * 1.05
ax[0,1].plot([0, mx], [0, mx], "crimson", ls="--", label="y = x (parity)")
for _, row in agg.iterrows():
    ax[0,1].annotate(f"L{int(row.L)}", (row.n_gt, row.emitted), fontsize=8, xytext=(4,4), textcoords="offset points")
ax[0,1].set(xlabel="n_gt (ground-truth contacts)", ylabel="mean contacts emitted", title="Mean contacts emitted vs GT count")
ax[0,1].legend(fontsize=8); fig.colorbar(sc, ax=ax[0,1], label="L")

# (3) recall & precision vs L
ax[1,0].plot(Ls, agg.recall, "o-", color="#4C72B0", label="recall (all, sep>=6)")
ax[1,0].plot(Ls, agg.precision, "^--", color="#C44E52", label="precision (all)")
ax[1,0].set(xlabel="protein length L", ylabel="mean per-rollout metric", title="Rollout accuracy vs length", ylim=(0, None))
ax[1,0].legend(fontsize=8)

# (4) generated tokens vs complete-doc tokens
full = 3 * agg.n_gt + 1
ax[1,1].scatter(full, agg.tokens, c=Ls, cmap="viridis", s=80, zorder=3)
mx = max(full.max(), agg.tokens.max()) * 1.05
ax[1,1].plot([0, mx], [0, mx], "crimson", ls="--", label="y = x (complete doc)")
ax[1,1].set(xlabel="tokens for a complete GT doc (3*n_gt+1)", ylabel="mean generated tokens", title="Document length: generated vs complete")
ax[1,1].legend(fontsize=8)

fig.suptitle(f"{MODEL_NICKNAME} - short-document-bias probe (your {N_ROLLOUTS} rollouts/protein)", fontsize=12)
fig.tight_layout(); plt.show()

## Takeaway

Every box in panel 1 and every point in panels 2 & 4 should sit **below parity** — the model emits
fewer contacts / shorter documents than the structure has — but the gap is modest for most proteins
and blows up only at the hard/long end, where panel 3 shows recall collapsing too. With a 100% finish
rate, that shortfall is the model choosing `<end>`, not a truncated generation. So forcing longer
documents (length penalty, banning early `<end>`, lower temperature) would mostly add **wrong**
contacts on exactly the proteins that under-generate — it isn't the lever; the fix is a model that
knows the fold.

- Issue & full writeup: [MarinFold #142](https://github.com/Open-Athena/MarinFold/issues/142)
- Code & offline analysis: [`experiments/exp142_evals_short_document_bias/`](https://github.com/Open-Athena/MarinFold/tree/main/experiments/exp142_evals_short_document_bias)